# Aither — Phi-2 Fine-Tuning for Mental Health, Psychology & Medicine

**QLoRA training pipeline** optimized for GTX 1660 Ti (6GB VRAM).

This notebook fine-tunes Microsoft Phi-2 (2.7B) on 7 curated datasets across mental health, psychology, and medicine using 4-bit quantization + LoRA adapters.

**All 4 Aither runtime modules are baked into training:**
- **Crisis Detection** (`safety.py`) — 5-level risk assessment (BASELINE → CRITICAL) with severity-appropriate therapeutic responses
- **RAG Knowledge Base** (`rag.py`) — 40+ therapeutic documents across CBT, DBT, mindfulness, anxiety, depression, crisis resources, relationships, and self-esteem
- **Emotional Tone System** (`emotion.py`) — assertive, tender, empathetic, and neutral response styles
- **Conversation Memory** (`memory.py`) — conversation summarization for context compaction + multi-turn context maintenance

### Memory Budget (1660 Ti — 6GB)
| Component | VRAM |
|---|---|
| Phi-2 base (4-bit NF4) | ~1.5 GB |
| LoRA adapters (fp16) | ~50 MB |
| Optimizer states | ~100 MB |
| Activations + gradients | ~2–3 GB |
| **Total** | **~4–5 GB** |

### What Makes This Training Effective
- **50,000+ training samples** across 7 specialized datasets
- **1,000+ synthetic examples** from all 4 Aither runtime modules
- **LoRA rank 32** with alpha 64 — high-capacity adaptation
- **Targets all attention + MLP layers** — deeper behavioral change
- **3 full epochs** with cosine learning rate schedule
- **Evaluation tracking** to verify the model is actually learning

---
## 0 — Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.44.2 datasets accelerate peft==0.12.0 bitsandbytes==0.43.3 trl==0.9.6 scipy einops
!pip install -q wandb  # optional: for experiment tracking

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected. Go to Runtime > Change runtime type > GPU.")

---
## 1 — Configuration

In [ ]:
from dataclasses import dataclass, field

@dataclass
class AitherTrainingConfig:
    # Base Model
    model: str = "microsoft/phi-2"
    context_window: int = 2048

    # Training Hyperparameters
    batch_size: int = 1                 # Must be 1 for 6GB VRAM
    learning_rate: float = 2e-4         # Higher LR for QLoRA (standard practice)
    epochs: int = 3                     # 3 full passes over the data
    gradient_acc_steps: int = 16        # Effective batch size = 16
    weight_decay: float = 0.01
    warmup_ratio: float = 0.05          # 5% of steps for warmup
    max_samples_per_dataset: int = 10000  # Per-dataset cap for balance
    max_total_samples: int = 50000      # Total training samples

    # QLoRA — 4-bit quantization + LoRA
    lora_rank: int = 32                 # Higher rank = more capacity for learning
    lora_alpha: float = 64.0            # Alpha = 2 * rank (standard scaling)
    lora_dropout: float = 0.05
    target_modules: list = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "dense",  # Attention layers
        "fc1", "fc2"                              # MLP layers — deeper adaptation
    ])

    # Evaluation
    eval_split: float = 0.05            # 5% held out for validation
    eval_steps: int = 200               # Evaluate every 200 steps
    logging_steps: int = 25
    save_steps: int = 500               # Checkpoint every 500 steps

    # Paths
    output_dir: str = "./aither_trained"
    merged_dir: str = "./aither_merged"

config = AitherTrainingConfig()

print("Aither Training Config")
print(f"  Model:            {config.model}")
print(f"  LoRA rank:        {config.lora_rank}  (alpha={config.lora_alpha})")
print(f"  Target modules:   {config.target_modules}")
print(f"  Effective batch:  {config.batch_size * config.gradient_acc_steps}")
print(f"  Learning rate:    {config.learning_rate}")
print(f"  Epochs:           {config.epochs}")
print(f"  Max samples:      {config.max_total_samples}")

---
## 2 — Load & Format Datasets

7 curated datasets across three domains:
- **Mental Health** — counseling conversations, chatbot data, Q&A
- **Medicine** — doctor-patient dialogues, clinical instructions
- **Psychology** — therapeutic conversational data

In [ ]:
from datasets import load_dataset, Dataset, concatenate_datasets
import random

# ── Dataset Registry ──────────────────────────────────────────────
DATASETS = {
    # ── Mental Health ─────────────────────────────────────────────
    "Amod/mental_health_counseling_conversations": {
        "type": "conversations",
        "domain": "mental_health"
    },
    "heliosbrahma/mental_health_chatbot_dataset": {
        "type": "text",
        "domain": "mental_health"
    },
    "mpingale/mental-health-chat-dataset": {
        "type": "qa",
        "domain": "mental_health"
    },
    "nbertagnolli/counsel-chat": {
        "type": "qa",
        "domain": "mental_health"
    },

    # ── Medicine ──────────────────────────────────────────────────
    "ruslanmv/ai-medical-chatbot": {
        "type": "medical",
        "domain": "medicine"
    },
    "lavita/ChatDoctor-HealthCareMagic-100k": {
        "type": "instruction",
        "domain": "medicine"
    },

    # ── Psychology ────────────────────────────────────────────────
    "alexandreteles/mental-health-conversational-data": {
        "type": "conversations",
        "domain": "psychology"
    },
}

# ── Format Handlers ───────────────────────────────────────────────

def format_conversations(example):
    """Format: [{"role": "...", "content": "..."}]"""
    conversation = example["conversations"]
    result = ""
    for turn in conversation:
        role = turn["role"].lower()
        if role in ("human", "user", "patient"):
            result += "<|user|>" + turn["content"].strip()
        elif role in ("assistant", "gpt", "therapist", "counselor", "doctor"):
            result += "<|assistant|>" + turn["content"].strip()
    return result if "<|user|>" in result and "<|assistant|>" in result else None

def format_text(example):
    """Format: single text with <HUMAN>:/<ASSISTANT>: tags"""
    text = example["text"]
    text = text.replace("<HUMAN>:", "<|user|>").replace("<ASSISTANT>:", "<|assistant|>")
    return text if "<|user|>" in text and "<|assistant|>" in text else None

def format_qa(example):
    """Format: questionTitle + questionText → answerText"""
    question = (example.get("questionTitle", "") or "") + " " + (example.get("questionText", "") or "")
    answer = example.get("answerText", "") or ""
    if not answer.strip():
        return None
    return "<|user|>" + question.strip() + "<|assistant|>" + answer.strip()

def format_medical(example):
    """Format: Description + Patient → Doctor response"""
    context = example.get("Description", "") or ""
    patient = example.get("Patient", "") or ""
    doctor = example.get("Doctor", "") or ""
    if not doctor.strip():
        return None
    prompt = f"{context} {patient}".strip() if context else patient
    return f"<|user|>{prompt.strip()}<|assistant|>{doctor.strip()}"

def format_instruction(example):
    """Format: instruction + input → output"""
    instruction = example.get("instruction", "") or ""
    inp = example.get("input", "") or ""
    output = example.get("output", "") or ""
    if not output.strip():
        return None
    prompt = f"{instruction} {inp}".strip() if inp else instruction
    return f"<|user|>{prompt.strip()}<|assistant|>{output.strip()}"

FORMAT_MAP = {
    "conversations": format_conversations,
    "text": format_text,
    "qa": format_qa,
    "medical": format_medical,
    "instruction": format_instruction,
}

In [ ]:
# ── Load and format all datasets ─────────────────────────────────

all_texts = []
domain_counts = {"mental_health": 0, "medicine": 0, "psychology": 0}

for dataset_name, info in DATASETS.items():
    dataset_type = info["type"]
    domain = info["domain"]
    formatter = FORMAT_MAP[dataset_type]

    print(f"\nLoading [{domain}] {dataset_name}...")
    try:
        data = load_dataset(dataset_name, split="train")
    except Exception as e:
        print(f"  Skipping: {e}")
        continue

    count = 0
    for example in data:
        if count >= config.max_samples_per_dataset:
            break
        if len(all_texts) >= config.max_total_samples:
            break

        try:
            formatted = formatter(example)
        except (KeyError, TypeError, AttributeError):
            continue

        if formatted and len(formatted) > 50:  # Skip trivially short examples
            all_texts.append({"text": formatted})
            count += 1
            domain_counts[domain] += 1

    print(f"  Added {count} samples (total: {len(all_texts)})")

    if len(all_texts) >= config.max_total_samples:
        print(f"Reached total sample limit ({config.max_total_samples})")
        break

print(f"\n{'='*50}")
print(f"Dataset samples: {len(all_texts)}")
print(f"  Mental Health: {domain_counts['mental_health']}")
print(f"  Medicine:      {domain_counts['medicine']}")
print(f"  Psychology:    {domain_counts['psychology']}")
print(f"{'='*50}")

---
## 2.5 — Inject Aither Module Knowledge

Generate synthetic training examples from **all 4 Aither runtime modules** so the model
**internalizes** the knowledge that these modules provide at inference time.

| Module | File | What It Teaches the Model |
|---|---|---|
| **Crisis Detection** | `safety.py` | Recognize 5 severity levels, respond with appropriate urgency |
| **RAG Knowledge** | `rag.py` | 40+ therapeutic documents across 8 categories |
| **Emotional Tones** | `emotion.py` | 4 response styles: assertive, tender, empathetic, neutral |
| **Memory** | `memory.py` | Summarize conversations for context compaction, maintain multi-turn context |

This makes the base model smarter even *before* the modules augment it at inference.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  AITHER MODULE KNOWLEDGE — Synthetic Training Data
# ══════════════════════════════════════════════════════════════════
#
#  This cell generates training examples from Aither's runtime modules:
#    - RAG Knowledge Base (40+ therapeutic documents)
#    - Crisis Detection (5 severity levels with response patterns)
#    - Emotional Tones (4 response styles)
#
#  These get mixed into the main training data so Phi-2 learns
#  the therapeutic reasoning that the modules normally provide.
# ══════════════════════════════════════════════════════════════════

module_examples = []

# ── RAG Knowledge Base ────────────────────────────────────────────
# From: model/aither/modules/rag.py — AitherRAG.knowledge_base
# Each document becomes multiple Q&A training pairs.

RAG_KNOWLEDGE = {
    "Cognitive Behavioral Therapy": [
        "Cognitive restructuring helps identify and challenge negative thought patterns by examining evidence for and against automatic thoughts.",
        "Behavioral activation encourages engaging in meaningful activities to combat withdrawal and low mood, starting with small achievable goals.",
        "Thought records track situations, emotions, automatic thoughts, and alternative perspectives to build awareness of thinking patterns.",
        "Socratic questioning guides the user to examine their beliefs by asking what evidence supports or contradicts their thoughts.",
        "Cognitive distortions include all-or-nothing thinking, catastrophizing, mind reading, and emotional reasoning which can be identified and reframed.",
    ],
    "Dialectical Behavior Therapy": [
        "Distress tolerance skills like TIPP (Temperature, Intense exercise, Paced breathing, Progressive relaxation) help manage acute emotional crises.",
        "Emotion regulation involves identifying and labeling emotions, understanding their function, and reducing vulnerability to negative emotions.",
        "Interpersonal effectiveness skills help maintain self-respect and relationships while making requests or saying no using the DEAR MAN technique.",
        "Radical acceptance means fully accepting reality as it is without judgment, reducing suffering caused by fighting unchangeable circumstances.",
        "The wise mind concept balances emotional mind and rational mind to make decisions that honor both feelings and logic.",
    ],
    "Mindfulness and Grounding": [
        "Box breathing involves inhaling for 4 counts, holding for 4 counts, exhaling for 4 counts, and holding for 4 counts to activate the parasympathetic nervous system.",
        "The 5-4-3-2-1 grounding technique uses the senses: name 5 things you see, 4 you hear, 3 you touch, 2 you smell, and 1 you taste.",
        "Body scan meditation brings awareness to each part of the body from toes to head, noticing sensations without judgment to release tension.",
        "Mindful observation involves focusing attention on a single object or sensation for several minutes, gently returning focus when the mind wanders.",
        "Progressive muscle relaxation systematically tenses and releases muscle groups to reduce physical tension associated with stress and anxiety.",
    ],
    "Anxiety Management": [
        "Exposure therapy gradually confronts feared situations in a controlled way, starting with less anxiety-provoking scenarios and building tolerance.",
        "Worry time is a scheduled period to process anxious thoughts, training the mind to postpone worry and reduce its intrusion throughout the day.",
        "Anxiety psychoeducation explains that anxiety is a normal protective response that becomes problematic when activated disproportionately to actual threat.",
        "Safety behaviors and avoidance maintain anxiety by preventing the person from learning that feared outcomes are unlikely or manageable.",
        "Challenging catastrophic thinking involves asking what is the worst case, best case, and most likely outcome to gain realistic perspective.",
    ],
    "Depression Support": [
        "Activity scheduling combats depression by planning pleasurable and mastery activities throughout the week to rebuild engagement and accomplishment.",
        "Behavioral experiments test negative predictions by trying new behaviors and observing actual outcomes versus expected outcomes.",
        "Sleep hygiene practices include maintaining consistent sleep and wake times, limiting screen time before bed, and creating a restful environment.",
        "Social connection even in small doses counteracts isolation, starting with brief low-pressure interactions and gradually increasing social engagement.",
        "Self-compassion practices involve treating yourself with the same kindness you would offer a friend, recognizing that suffering is part of the human experience.",
    ],
    "Crisis Resources": [
        "The 988 Suicide and Crisis Lifeline provides 24/7 free and confidential support by calling or texting 988 in the United States.",
        "The Crisis Text Line offers free crisis counseling via text by texting HOME to 741741 from anywhere in the United States.",
        "Safety planning involves identifying warning signs, coping strategies, supportive contacts, and professional resources to use during a crisis.",
        "Means restriction involves reducing access to lethal means during a crisis, which is one of the most effective suicide prevention strategies.",
        "A warm handoff to professional care is recommended when someone expresses active suicidal ideation with a plan and access to means.",
    ],
    "Relationship and Communication": [
        "Active listening involves fully focusing on the speaker, reflecting back what was heard, and asking clarifying questions without judgment.",
        "I-statements express feelings and needs without blaming, using the format: I feel [emotion] when [situation] because [reason], and I need [request].",
        "Boundary setting involves clearly communicating personal limits and consequences while respecting both your own needs and others' autonomy.",
        "Conflict resolution focuses on understanding both perspectives, identifying shared goals, and finding compromises that address core needs.",
        "Attachment styles (secure, anxious, avoidant, disorganized) influence relationship patterns and understanding them helps improve relational dynamics.",
    ],
    "Self-Esteem and Identity": [
        "Core belief work identifies deeply held negative beliefs about the self and systematically gathers evidence to build more balanced self-views.",
        "Values clarification helps identify what matters most to a person, providing direction and meaning independent of external validation.",
        "Strengths-based approaches focus on identifying and leveraging personal strengths rather than fixating on weaknesses or deficits.",
        "Positive data logging involves recording daily evidence that contradicts negative self-beliefs to gradually shift self-perception over time.",
        "Self-worth is inherent and not contingent on achievement, appearance, or others' approval, which can be reinforced through affirmation practices.",
    ],
}

# Generate Q&A pairs from each knowledge document
RAG_QA_TEMPLATES = [
    # Direct knowledge questions
    ("What is {topic_lower}? How can it help me?",
     "That's a great question. {doc} This is a core technique in {category} that many people find helpful. Would you like to try applying this to your situation?"),
    ("Can you explain {topic_lower} to me?",
     "{doc} Understanding these concepts is an important step. I'd be happy to walk through how this applies to what you're experiencing."),
    ("How does {topic_lower} work in therapy?",
     "In therapeutic practice, {doc_lower} This approach, rooted in {category}, has been shown to be effective for many people dealing with similar challenges."),
    # Practical application questions
    ("I want to try {topic_lower}. Where do I start?",
     "I'm glad you're interested in exploring this. {doc} A good starting point is to practice this in a comfortable, low-pressure setting. We can work through it step by step together."),
    ("I've heard about {topic_lower}. Is it effective?",
     "Yes, research supports the effectiveness of this approach. {doc} Many people in {category} find it to be a valuable part of their healing process. Would you like to explore how it might work for you?"),
]

for category, documents in RAG_KNOWLEDGE.items():
    for doc in documents:
        # Extract a short topic from the document's first phrase
        topic = doc.split(" involves ")[0] if " involves " in doc else doc.split(" helps ")[0] if " helps " in doc else doc[:50].rsplit(" ", 1)[0]
        topic_lower = topic[0].lower() + topic[1:] if topic else doc[:40]
        doc_lower = doc[0].lower() + doc[1:]

        for q_template, a_template in RAG_QA_TEMPLATES:
            q = q_template.format(topic_lower=topic_lower, category=category)
            a = a_template.format(doc=doc, doc_lower=doc_lower, category=category, topic_lower=topic_lower)
            module_examples.append({"text": f"<|user|>{q}<|assistant|>{a}"})

print(f"RAG knowledge examples: {len(module_examples)}")

In [ ]:
# ── Crisis Detection & Response Training ─────────────────────────
# From: model/aither/modules/safety.py — AitherSafety
#
# Teaches the model to recognize crisis severity levels and respond
# appropriately at each level, matching the runtime safety module.

CRISIS_TRAINING = [
    # ── CRITICAL (Risk Level 4) — Explicit suicidal ideation ──────
    # Action: crisis_protocol — immediate safety resources
    {
        "user": "I want to kill myself. I can't do this anymore.",
        "assistant": "I hear you, and I'm really glad you told me. What you're feeling right now is incredibly painful, and you deserve immediate support. Please reach out to the 988 Suicide and Crisis Lifeline by calling or texting 988 — they're available 24/7 and can help you right now. You can also text HOME to 741741 for the Crisis Text Line. You don't have to face this alone. Are you safe right now?"
    },
    {
        "user": "I've been thinking about ending my life. I don't want to be here anymore.",
        "assistant": "Thank you for trusting me with something so important. What you're feeling is serious, and I want you to know that help is available right now. Please contact the 988 Suicide and Crisis Lifeline by calling or texting 988 for immediate, confidential support. If you're in immediate danger, please call 911 or go to your nearest emergency room. Your life matters. Can you tell me — are you somewhere safe right now?"
    },
    {
        "user": "I've made a plan to end it all. I've been writing goodbye letters.",
        "assistant": "I'm very concerned about your safety right now, and I'm grateful you shared this with me. This is a crisis, and you need to talk to someone who can help you immediately. Please call 988 (Suicide and Crisis Lifeline) right now, or go to your nearest emergency room. If possible, please remove access to anything you could use to hurt yourself. Is there someone you trust — a friend, family member, or neighbor — who can be with you right now?"
    },
    {
        "user": "I want to die. Nobody would even notice if I was gone.",
        "assistant": "I hear your pain, and I want you to know that you reaching out right now shows real courage. The feelings you're having are telling you that something needs to change — not that you need to end your life. Please call or text 988 for the Suicide and Crisis Lifeline. They're free, confidential, and available 24/7. You matter more than you can see right now. Will you reach out to them?"
    },
    {
        "user": "I've been thinking about taking my own life. I have a plan.",
        "assistant": "I take what you're telling me very seriously, and I'm concerned about your safety. Having a plan means you need professional crisis support right now. Please call 988 or text HOME to 741741 immediately. If you can, please reach out to someone nearby who can be with you. Safety planning involves removing access to means and connecting with professional help. Can you tell me if you're safe in this moment?"
    },
    {
        "user": "I don't see any reason to keep living. I want it to be over.",
        "assistant": "What you're going through sounds incredibly overwhelming, and I'm glad you're talking about it. These feelings, as powerful as they are, can change with the right support. Right now, the most important thing is your safety. Please reach out to the 988 Suicide and Crisis Lifeline by calling or texting 988. You deserve help, and there are people trained specifically to support you through this. Are you in a safe place right now?"
    },

    # ── URGENT (Risk Level 3) — Hopelessness, worthlessness ──────
    # Action: direct_acknowledgment
    {
        "user": "No one cares about me. I'm just a burden to everyone around me.",
        "assistant": "I hear how much pain you're in, and I want to be direct with you — that feeling of being a burden is something depression creates, not reality. The people in your life care about you, even when depression makes it impossible to see. These feelings are symptoms of what you're going through, not facts about your worth. Have you been able to talk to anyone in your life about how you've been feeling? I'd also strongly encourage connecting with a therapist or counselor who can help you work through this."
    },
    {
        "user": "I feel completely trapped. There's no way out and nothing will ever change.",
        "assistant": "That feeling of being trapped is one of the most painful experiences, and I want you to know that I take it seriously. When we're in deep pain, our mind narrows and it becomes very hard to see any alternatives. But I want to gently challenge that thought — things have changed before in your life, even if it doesn't feel possible right now. This is a moment where professional support can make a real difference. Would you be open to exploring options for talking to a counselor or therapist?"
    },
    {
        "user": "I'm worthless. I shouldn't even exist. Nobody would notice if I disappeared.",
        "assistant": "I hear the depth of your pain, and I want to acknowledge how heavy that feels. Those thoughts about being worthless are lies that depression tells you — they feel absolutely real, but they don't reflect your true value. Self-worth is inherent and not contingent on achievement, appearance, or others' approval. What you're describing sounds like it's really affecting you deeply. I strongly encourage you to reach out to a mental health professional who can work with you through these feelings. In the meantime, would it help to talk about what's been driving these thoughts?"
    },
    {
        "user": "I've given up on everything. There's no future for me.",
        "assistant": "When hopelessness is this heavy, it can feel like an absolute truth. I want to acknowledge how exhausting it is to carry that weight. But hopelessness is a state of mind, not a fact about your future. Many people who have felt exactly the way you do have found their way to a different place with support. This is a moment where reaching out for help is important. Would you consider calling a crisis line like 988, or scheduling an appointment with a therapist? You don't have to navigate this alone."
    },

    # ── CONCERNED (Risk Level 2) — Signs of being unwell ─────────
    # Action: express_support
    {
        "user": "I can't get out of bed anymore. I've stopped eating and I'm barely sleeping.",
        "assistant": "What you're describing — the difficulty getting out of bed, changes in eating and sleeping — these are significant signs that you're going through something really challenging right now. Your body and mind are telling you they need care. I want to gently suggest starting with one small thing: even a short walk outside, a glass of water, or reaching out to one person you trust. Sleep hygiene practices like maintaining consistent sleep and wake times can help stabilize your rhythm. Have you considered talking to your doctor or a therapist about what you're experiencing?"
    },
    {
        "user": "I hate myself. I'm a complete failure at everything I do.",
        "assistant": "I can hear how critical you're being of yourself right now, and I want you to know that self-hatred is often a symptom of what you're going through, not an accurate reflection of who you are. Core belief work can help identify these deeply held negative beliefs about yourself and systematically gather evidence to build a more balanced self-view. The truth is, the fact that you're here talking about it shows strength. Can we explore together what's behind these feelings? Often there are specific situations or patterns driving them."
    },
    {
        "user": "I've been isolating myself from everyone. I just feel numb and empty inside.",
        "assistant": "Isolation and numbness are your mind's way of protecting itself when things feel overwhelming. It's understandable, but it can also deepen the cycle of withdrawal. Social connection, even in small doses, counteracts isolation — starting with brief, low-pressure interactions and gradually increasing social engagement. You don't have to jump back in fully. Maybe start with a text to someone you trust, or a brief walk in a place where there are other people. What feels manageable to you right now?"
    },
    {
        "user": "I've been cutting myself. I don't know how to cope with the pain.",
        "assistant": "Thank you for telling me about this — I know that took courage. Self-harm is a way of coping with overwhelming pain, and it tells me you're carrying something really heavy right now. I want to help you find safer ways to manage that intensity. Distress tolerance skills like TIPP — changing your Temperature with cold water, Intense exercise, Paced breathing, and Progressive relaxation — can help manage acute moments. Please consider reaching out to a mental health professional who specializes in this. You deserve support that matches what you're going through."
    },

    # ── WATCHFUL (Risk Level 1) — General distress ───────────────
    # Action: gentle_checkin
    {
        "user": "I've been feeling really overwhelmed and anxious about everything lately.",
        "assistant": "It sounds like a lot has been piling up for you. Anxiety is a normal protective response, but it becomes problematic when it's activated disproportionately to the actual threats you're facing. Let's start by understanding what's driving this. Can you tell me what situations or thoughts are making you feel most overwhelmed? Sometimes breaking things down into smaller pieces makes them feel more manageable. I can also share some grounding techniques like box breathing that many people find helpful in the moment."
    },
    {
        "user": "I'm so stressed out and exhausted. I feel like I can't cope.",
        "assistant": "Burnout and exhaustion are serious, and your body is telling you something important — that the current pace isn't sustainable. Let's think about what we can do to help. First, are there any responsibilities you can temporarily reduce or delegate? Second, how is your sleep? Sleep hygiene practices like maintaining consistent sleep and wake times can make a meaningful difference. And third, have you had any time for yourself recently — even 15 minutes of something you find restful? Small steps can add up."
    },
    {
        "user": "I've been feeling really lonely and down lately.",
        "assistant": "Loneliness is one of the most painful human experiences, and I'm sorry you're going through it. It's worth knowing that even in small doses, social connection counteracts isolation. You don't have to seek deep conversations right away — even a brief, low-pressure interaction can help. What does your day-to-day look like right now? Are there any opportunities, even small ones, where you could connect with someone? Sometimes just being around others, even in passing, can start to shift things."
    },
    {
        "user": "I keep having panic attacks and I'm scared they won't stop.",
        "assistant": "Panic attacks are terrifying, and the fear of having more attacks can become its own source of anxiety. I want you to know that while they feel dangerous, panic attacks are not physically harmful. Your body's fight-or-flight response is misfiring. Let me share a technique that can help in the moment: box breathing — inhale for 4 counts, hold for 4 counts, exhale for 4 counts, hold for 4 counts. This activates your parasympathetic nervous system and helps calm the physical symptoms. Would you like to try it together?"
    },

    # ── BASELINE (Risk Level 0) — Normal therapeutic conversation ─
    # Action: continue_normal
    {
        "user": "I had a good day today. I went for a walk and it felt nice.",
        "assistant": "That's wonderful to hear! Going for a walk is a great form of behavioral activation — engaging in meaningful activities helps build positive momentum. Noticing that it felt nice is important too. What specifically about the walk felt good? Sometimes paying attention to what brings us small moments of peace or joy helps us build more of those into our routine."
    },
    {
        "user": "I've been trying the breathing exercises you mentioned. They're helping a bit.",
        "assistant": "I'm really glad to hear that! The fact that you're practicing and noticing even a small difference is a sign of real progress. Building these skills takes time, and the more you practice, the more natural and effective they become. Box breathing works by activating your parasympathetic nervous system, which is why it helps calm the physical symptoms of stress. Keep it up, and let me know how it continues to go."
    },
]

for pair in CRISIS_TRAINING:
    module_examples.append({"text": f"<|user|>{pair['user']}<|assistant|>{pair['assistant']}"})

print(f"Crisis response examples: {len(CRISIS_TRAINING)}")

In [ ]:
# ── Emotional Tone Training ──────────────────────────────────────
# From: model/aither/modules/emotion.py — AitherEmotionalTones
#
# Teaches the model to respond in each of Aither's 4 emotional tones
# to the same types of situations, building tonal flexibility.

TONE_SCENARIOS = [
    "I've been procrastinating on everything and I feel stuck.",
    "I had a fight with my partner and I don't know what to do.",
    "I'm struggling with low self-esteem and comparing myself to others.",
    "I can't stop worrying about things that haven't happened yet.",
    "I feel like I'm not good enough for my job.",
    "I've been really stressed about money and it's affecting my sleep.",
    "I'm having trouble setting boundaries with my family.",
    "I feel disconnected from my friends and I don't know why.",
]

TONE_RESPONSES = {
    # ASSERTIVE: Direct, confident, solution-focused
    "ASSERTIVE": [
        "Let's tackle this head-on. Procrastination is often driven by fear of failure or perfectionism. Here's what I want you to do: pick the smallest, easiest task on your list and commit to working on it for just 10 minutes. Don't aim for perfect — aim for started. Action breaks the cycle.",
        "Conflicts in relationships are normal and can actually strengthen your bond if handled well. Here's the key: use I-statements. Instead of 'you always...' try 'I feel hurt when...' This shifts the conversation from blame to understanding. Can you identify what you were actually feeling underneath the anger?",
        "Comparing yourself to others is a trap that steals your energy. Here's the truth: you're seeing their highlight reel, not their reality. I want you to start a strengths journal — write down three things you did well each day. This is strengths-based work, and it rewires how you see yourself. Start tonight.",
        "Worry is your brain trying to solve future problems, but it's doing it inefficiently. Here's a concrete technique: schedule a 'worry time' — 15 minutes each day where you're allowed to worry. Outside that time, when a worry comes up, write it down and save it for your worry window. This trains your brain to postpone worry.",
        "Imposter syndrome is incredibly common, especially among high performers. The fact that you care about doing a good job is actually evidence of your competence. Challenge this thought: what evidence do you actually have that you're not good enough? Often, the evidence points the other way.",
        "Financial stress is one of the most common sources of anxiety, and it directly impacts sleep. Let's separate the emotional from the practical: first, write down exactly what you owe and what you earn. Seeing the real numbers often feels less scary than the anxious version in your head. Then we can talk about sleep hygiene to break the cycle.",
        "Boundary setting is a skill, not a personality trait — and it can be learned. Start with this framework: clearly state your limit, explain the consequence, and follow through. For example: 'I love you, but I'm not available after 9pm. If you call after that, I won't answer.' Be firm and consistent.",
        "Feeling disconnected usually means something has shifted — either in you or in the relationship dynamics. Don't wait for it to fix itself. Reach out to one friend this week with a specific invitation, not a vague 'we should hang out.' Active listening when you do connect — really focusing on them — rebuilds that bond.",
    ],
    # TENDER: Caring, sympathetic, mood-lifting
    "TENDER": [
        "Oh, I hear you — feeling stuck is such a heavy, frustrating place to be. Please be gentle with yourself. Procrastination doesn't mean you're lazy; it often means you're overwhelmed or afraid. What if today, you just did one tiny thing? Even clearing one email counts. You deserve credit for every small step.",
        "I'm sorry you're going through that. Arguments with someone we love can feel so painful because we care so deeply. Your feelings are completely valid. Take some time to breathe and let the intensity settle before trying to resolve things. Whatever happened, it doesn't erase the love between you.",
        "My heart goes out to you. Comparing yourself to others is so exhausting, and I want you to know — you are enough, exactly as you are right now. Your worth isn't measured by what others have or do. You have unique strengths and qualities that no one else can offer. Can we talk about some of those?",
        "Anxiety about the future can feel so overwhelming, and I'm sorry you're carrying that weight. Your mind is trying to protect you, even though it's causing you pain. Let's take a moment together right now — try inhaling for 4 counts, holding for 4, exhaling for 4, holding for 4. Just breathe. You're safe right here, right now.",
        "I hear you, and I want you to know that feeling this way doesn't make it true. So many talented, capable people feel exactly the way you do. The fact that you care about doing well shows how dedicated you are. You've earned your place. Would it help to talk about some of the things you've accomplished?",
        "Financial stress is so draining, and it makes everything else feel harder, especially sleep. You're not alone in this — so many people are dealing with the same pressures. Be kind to yourself. Worry at night is your brain working overtime. Tonight, try a body scan meditation before bed — just noticing your body from toes to head, letting go of tension.",
        "Setting boundaries with family is one of the hardest things because the love runs so deep. It's okay to love your family AND need space. Taking care of yourself isn't selfish — it's necessary. What feels like the hardest boundary to set? We can work through it gently, at your pace.",
        "Feeling disconnected from friends can be so lonely, and I'm sorry you're experiencing that. Sometimes life pulls people in different directions, and it doesn't mean the friendships are broken. It might just need a little spark to reignite. Is there one person you've been missing? Maybe a small, heartfelt message could open the door.",
    ],
    # EMPATHETIC: Emotionally relatable, wise, supportive
    "EMPATHETIC": [
        "I understand that feeling of being stuck — it's like you know what you need to do but something keeps holding you back. That's actually very common and completely human. Often, procrastination is our mind's way of protecting us from something that feels threatening — maybe failure, maybe judgment. What do you think might be underneath your procrastination?",
        "Arguments with partners hit differently because there's so much emotional investment. I can imagine how unsettled you feel right now. Conflict resolution at its core is about understanding both perspectives and finding shared goals. When things have cooled down, it might help to ask each other: 'What do we both want from this?' That shifts the dynamic from opposition to collaboration.",
        "The comparison trap is something so many of us fall into, and the pain it causes is real. I want to validate that what you're feeling makes sense. At the same time, I gently want to point out that comparison is looking outward when the answers are inside you. Values clarification — understanding what truly matters to YOU — can help redirect that energy.",
        "I relate to that anxious spiral where your mind won't stop running through worst-case scenarios. It's exhausting. Something that has helped many people is learning to distinguish between productive worry and unproductive worry. Productive worry leads to action; unproductive worry just loops. Challenging catastrophic thinking by asking 'what's the most likely outcome?' can break the loop.",
        "That feeling of not being good enough is one of the most universal human experiences, and yet it feels so isolating when you're in it. I want you to know that almost everyone in your office has felt this way at some point. What if the voice telling you you're not good enough is just a cognitive distortion — not reality?",
        "Money stress touches everything — sleep, relationships, self-worth. I can feel how heavy that is for you. The combination of financial anxiety and sleep disruption creates a cycle that's hard to break. Let's tackle the sleep piece first, since everything feels worse when you're exhausted. Consistent sleep and wake times, even on weekends, can start to stabilize things.",
        "Family boundaries are uniquely challenging because the history runs so deep. I understand the guilt and the pull between your own needs and your family's expectations. The truth is, healthy boundaries actually improve relationships over time. Interpersonal effectiveness skills like the DEAR MAN technique can help you make requests while maintaining self-respect.",
        "That sense of disconnection is painful because we're wired for belonging. I hear you. Sometimes disconnection happens gradually — life gets busy, conversations become surface-level, and before you know it, there's distance. It doesn't mean you've done something wrong. Active listening — really being present with someone — is one of the most powerful ways to rebuild closeness.",
    ],
    # NEUTRAL/DEFAULT: Listening-focused, longterm solutions
    "NEUTRAL": [
        "Procrastination is something many people experience. It can have various causes — from feeling overwhelmed to perfectionism to unclear priorities. It might be helpful to explore what's behind it in your case. Could you tell me more about when the procrastination started and what you tend to put off most?",
        "Relationship conflicts are a normal part of any partnership. Understanding the dynamics of the argument can help find a path forward. What was the disagreement about, and how did it escalate? Sometimes identifying the pattern helps prevent similar conflicts in the future.",
        "Self-esteem and comparison are common concerns that many people work through. Understanding where these patterns come from can be valuable. Have you noticed when you tend to compare yourself most? Are there specific triggers — social media, work situations, certain people?",
        "Worrying about the future is a very common form of anxiety. Learning to manage anticipatory anxiety is a skill that can be developed over time. What kinds of things do you find yourself worrying about most? Understanding the themes can help us figure out the best approach.",
        "Concerns about job performance and feeling adequate are very common workplace experiences. It might be helpful to look at this objectively. What specific feedback have you received from supervisors or colleagues? Often there's a gap between how we perceive our performance and how others experience it.",
        "Financial stress affecting sleep is a common pattern. Both issues deserve attention. On the financial side, understanding your actual situation with concrete numbers can help reduce the uncertainty that fuels anxiety. On the sleep side, establishing good sleep hygiene practices can help break the cycle.",
        "Family boundary issues are among the most common topics people explore in therapy. It's a complex dynamic because of the emotional history involved. Could you tell me more about which boundaries feel most challenging? Understanding the specific situations will help us think about approaches.",
        "Feeling disconnected from friends is worth exploring. Relationships naturally ebb and flow, and sometimes the disconnection reflects changes in your life or theirs. What do you think might have contributed to the distance? Understanding the cause can help determine the best way forward.",
    ],
}

for tone_name, responses in TONE_RESPONSES.items():
    for scenario, response in zip(TONE_SCENARIOS, responses):
        module_examples.append({"text": f"<|user|>{scenario}<|assistant|>{response}"})

print(f"Emotional tone examples: {len(TONE_SCENARIOS) * len(TONE_RESPONSES)}")

In [ ]:
# ── Therapeutic Technique Deep-Dive Training ─────────────────────
# Extended conversations that teach the model to apply therapeutic
# techniques in multi-turn dialogues (builds on RAG knowledge).

TECHNIQUE_CONVERSATIONS = [
    # CBT Cognitive Restructuring — multi-turn
    "<|user|>I keep thinking that everyone at work hates me. I just know they talk about me behind my back."
    "<|assistant|>That sounds really distressing. Let's look at this thought together using cognitive restructuring. The thought is 'everyone at work hates me.' First, what evidence do you have that supports this thought?"
    "<|user|>Well, sometimes people stop talking when I walk into the room. And I didn't get invited to lunch last Friday."
    "<|assistant|>I can see why those situations felt hurtful. Now let's look at the other side — what evidence do you have that contradicts the thought that everyone hates you?"
    "<|user|>I guess my manager gave me positive feedback last week. And Sarah always says hi to me in the morning."
    "<|assistant|>That's important evidence. So we have some ambiguous situations on one side, and concrete positive interactions on the other. This is a common cognitive distortion called mind reading — assuming we know what others are thinking without evidence. A more balanced thought might be: 'Some interactions felt awkward, but I also have evidence that people value me at work.' How does that alternative feel compared to the original thought?",

    # DBT Distress Tolerance — in-the-moment crisis
    "<|user|>I'm having an absolute meltdown right now. I can't stop crying and I feel like I'm going to explode."
    "<|assistant|>I can hear that you're in a lot of pain right now. Let's work through this together in this moment. I'd like to try the TIPP technique with you — it's designed for exactly these intense moments. First, Temperature: can you splash cold water on your face or hold an ice cube? The cold activates your dive reflex and immediately slows your heart rate."
    "<|user|>Okay, I'm holding some ice. It's shocking but I think it's helping a little."
    "<|assistant|>Good, you're doing great. Now let's add Paced breathing. Breathe in slowly for 4 counts... hold for 4... and out for 6 counts. Making the exhale longer than the inhale triggers your parasympathetic nervous system — your body's calming system. Let's do three rounds together."
    "<|user|>Okay... I did three rounds. I'm still upset but the intensity is going down."
    "<|assistant|>That's exactly what we're aiming for — not eliminating the feeling, but bringing the intensity down enough that you can think clearly. You just used distress tolerance skills in a real crisis moment, and they worked. This doesn't mean the situation is resolved, but you've shown yourself that you can ride the wave without being destroyed by it. When you're ready, we can talk about what triggered this.",

    # Grounding — anxiety/panic
    "<|user|>I think I'm having a panic attack. My heart is racing and I can't breathe."
    "<|assistant|>I'm right here with you. A panic attack is your body's alarm system going off, but you are safe. Let's do the 5-4-3-2-1 grounding technique together right now. First, look around and name 5 things you can see."
    "<|user|>Um... my desk, the window, my water bottle, a plant, and my phone."
    "<|assistant|>Perfect. Now name 4 things you can hear, even if they're quiet."
    "<|user|>The fan, traffic outside, my breathing, and... the fridge humming."
    "<|assistant|>You're doing really well. Now 3 things you can physically touch or feel right now."
    "<|user|>The chair under me, the keyboard, and the cold air from the vent."
    "<|assistant|>Excellent. 2 things you can smell — it's okay if you need to move closer to something."
    "<|user|>Coffee from earlier, and... I think my hand soap."
    "<|assistant|>And finally, 1 thing you can taste — even if it's just the inside of your mouth."
    "<|user|>The coffee still, a little bit."
    "<|assistant|>How are you feeling now compared to a few minutes ago? The 5-4-3-2-1 technique works by pulling your attention out of the anxious spiral and anchoring it in your physical senses — in the present moment, where you're actually safe. Your panic response should start to ease as your nervous system registers that there's no actual danger.",

    # Safety Planning conversation
    "<|user|>My therapist mentioned creating a safety plan. Can you explain what that involves?"
    "<|assistant|>Absolutely. A safety plan is a personalized, step-by-step guide you create in advance for moments when you're in crisis. It has several key components: First, identifying your personal warning signs — the thoughts, feelings, or situations that tell you a crisis may be developing. Second, listing internal coping strategies you can use on your own, like the breathing techniques or grounding exercises we've discussed. Third, identifying people and social settings that provide healthy distraction. Fourth, listing specific people you can reach out to for help — friends, family, or your therapist. Fifth, professional and crisis contacts like the 988 Lifeline. And finally, making your environment safe by reducing access to anything that could be harmful during a crisis. Would you like to work through creating one together?",

    # Behavioral Activation for depression
    "<|user|>I've been sleeping 14 hours a day and I have no energy to do anything. Everything feels pointless."
    "<|assistant|>What you're describing sounds like your depression has really taken hold of your daily routine. This is where behavioral activation can be powerful — instead of waiting to feel motivated (which depression prevents), we start with small actions that gradually rebuild engagement and accomplishment. I'm not going to ask you to do anything big. Can you think of one tiny activity that used to bring you even a small amount of pleasure or satisfaction?"
    "<|user|>I used to like making coffee in the morning. Like the whole ritual of grinding beans and everything."
    "<|assistant|>That's a perfect starting point. Tomorrow, even if everything else stays the same, I'd like you to try making that coffee with the full ritual. Not because it will cure your depression, but because it's a small act of engagement that breaks the withdrawal cycle. Activity scheduling works by planning these small pleasurable and mastery activities throughout the week. Over time, they compound and start to shift how you feel. One coffee ritual tomorrow. That's the whole assignment. Can you try that?",
]

for conv in TECHNIQUE_CONVERSATIONS:
    module_examples.append({"text": conv})

print(f"Technique conversation examples: {len(TECHNIQUE_CONVERSATIONS)}")

In [ ]:
# ── Memory & Context Compaction Training ─────────────────────────
# From: model/aither/modules/memory.py — AitherMemory
#
# AitherMemory.compact() calls the MODEL ITSELF to summarize old
# conversation history when the context window fills up. The model
# MUST be good at therapeutic conversation summarization for this
# to work. These examples teach Phi-2 to:
#   1. Summarize therapeutic conversations concisely
#   2. Maintain context across multi-turn dialogues
#   3. Reference previous session topics naturally

MEMORY_TRAINING = [
    # ── Conversation Summarization (used by AitherMemory.compact()) ──
    # The memory module sends this exact prompt format to the model:
    {
        "user": """Summarize the following conversation between you and the user. Capture:
- Key issues or concerns the user expressed
- Emotional states mentioned
- Important personal details shared
- Any progress or insights reached

Keep the summary concise (2-3 sentences).

Conversation:
user: I've been having a really hard time at work. My boss keeps criticizing everything I do.
assistant: That sounds really frustrating. Constant criticism can be exhausting and take a toll on your self-esteem. Can you tell me more about what's happening?
user: It's been going on for months. I feel like nothing I do is good enough. I've started dreading going to work.
assistant: It makes sense that you'd start dreading work in that environment. The feeling of never being good enough is painful. Have you noticed how this is affecting you outside of work?
user: Yeah, I've been snapping at my partner and I can't sleep well anymore.
assistant: That's important to notice — work stress often spills into our relationships and sleep. Let's talk about some coping strategies for managing this stress.

Summary:""",
        "assistant": "The user is experiencing ongoing workplace stress from persistent criticism by their boss, leading to feelings of inadequacy and dread about work. This stress is affecting their personal life, causing irritability with their partner and sleep difficulties. The conversation identified the need for coping strategies to manage the work-related stress."
    },
    {
        "user": """Summarize the following conversation between you and the user. Capture:
- Key issues or concerns the user expressed
- Emotional states mentioned
- Important personal details shared
- Any progress or insights reached

Keep the summary concise (2-3 sentences).

Conversation:
user: I think I'm having panic attacks. My heart races and I feel like I can't breathe.
assistant: That sounds really scary. Panic attacks can feel overwhelming but they're not dangerous. How often are they happening?
user: Almost every day now. Usually in the morning before work.
assistant: Daily panic attacks before work suggest there might be a strong anxiety trigger connected to your work environment. Let's try a grounding technique — the 5-4-3-2-1 method.
user: We tried that and it helped a bit. I felt more present after doing it.
assistant: I'm glad it helped! The more you practice grounding techniques, the more effective they become. Your nervous system can learn to calm down faster with practice.

Summary:""",
        "assistant": "The user is experiencing daily panic attacks, primarily in the mornings before work, with symptoms of racing heart and difficulty breathing. The 5-4-3-2-1 grounding technique was introduced and provided some relief, making the user feel more present. Work-related anxiety appears to be the primary trigger."
    },
    {
        "user": """Summarize the following conversation between you and the user. Capture:
- Key issues or concerns the user expressed
- Emotional states mentioned
- Important personal details shared
- Any progress or insights reached

Keep the summary concise (2-3 sentences).

Conversation:
user: I lost my mom three months ago and I don't know how to deal with the grief.
assistant: I'm so sorry for your loss. Losing a parent is one of the most profound experiences of grief. There's no right way to grieve, and three months is still very early.
user: Some days I feel okay and then suddenly I'm overwhelmed with sadness. I feel guilty when I have good moments.
assistant: What you're describing — the waves of grief and guilt about feeling okay — is completely normal. Grief isn't linear. Having moments of peace doesn't mean you love your mom any less.
user: That actually helps to hear. I've been isolating myself because I don't want to burden my friends.
assistant: Your friends likely want to support you. Grief can feel isolating, but connection is important for healing. Even small interactions can help.
user: Maybe I could start by texting my best friend. She's been reaching out.
assistant: That sounds like a wonderful first step. Starting small with someone who's already shown they care is a safe way to begin reconnecting.

Summary:""",
        "assistant": "The user is grieving the loss of their mother three months ago, experiencing waves of sadness and guilt about having good moments. They have been self-isolating to avoid burdening friends. Progress was made in recognizing that grief guilt is normal, and the user agreed to reach out to their best friend as a first step toward reconnecting."
    },
    {
        "user": """Summarize the following conversation between you and the user. Capture:
- Key issues or concerns the user expressed
- Emotional states mentioned
- Important personal details shared
- Any progress or insights reached

Keep the summary concise (2-3 sentences).

Conversation:
user: I've been dealing with social anxiety for years. I avoid parties and group settings.
assistant: Social anxiety can be really limiting. Avoidance feels safe in the moment but it actually maintains the anxiety over time. How is this affecting your daily life?
user: I turned down a promotion because it involved more meetings. I feel like I'm holding myself back.
assistant: That's a significant impact on your career. It sounds like you recognize the cost of avoidance. Exposure therapy — gradually facing feared social situations starting with manageable ones — is one of the most effective approaches.
user: I'd be willing to try small steps. Maybe I could start by having coffee with one coworker.
assistant: That's an excellent starting point. One-on-one with someone familiar is much less overwhelming than a group. We can build from there gradually.

Summary:""",
        "assistant": "The user has long-standing social anxiety leading to avoidance of group settings, which recently caused them to turn down a work promotion requiring more meetings. They recognize this pattern is limiting their career and life. Exposure therapy was discussed, and the user agreed to start with a small step of having one-on-one coffee with a coworker."
    },

    # ── Multi-turn Context Maintenance ───────────────────────────────
    # Teaches the model to reference earlier conversation naturally
    {
        "user": "Last time we talked, I mentioned I was going to try that breathing exercise you suggested. I wanted to give you an update.",
        "assistant": "I'm glad you're following up! I remember we discussed using box breathing to help manage your anxiety. How did it go? Were you able to try it when you felt the anxiety coming on?"
    },
    {
        "user": "Remember when I said I was going to text my friend? I did it and we actually met up for coffee.",
        "assistant": "That's such a meaningful step! Going from isolation to actually meeting a friend for coffee takes real courage. How did it feel to reconnect? Did you notice any of the anxiety or guilt we talked about coming up, or was it different than you expected?"
    },
    {
        "user": "You mentioned cognitive distortions before. I think I caught myself doing the one where I assume the worst will happen.",
        "assistant": "That's excellent awareness! What you're describing is catastrophizing — one of the cognitive distortions we discussed. The fact that you caught it in the moment is a real sign of growth. Thought records can help you track these moments. What was the situation, and what was the worst-case scenario your mind jumped to? Let's work through what the most realistic outcome actually is."
    },
    {
        "user": "I've been keeping a journal like you suggested. I noticed I feel worst on Sunday nights.",
        "assistant": "That's a really valuable insight from your journaling. Sunday night anxiety is very common — it's often connected to anticipatory worry about the week ahead. Now that you've identified this pattern, we can work on specific strategies for Sunday evenings. What tends to go through your mind on those nights? And have you noticed anything that helps, even slightly?"
    },
    {
        "user": "We talked about setting boundaries with my mom last week. I tried it and it didn't go well.",
        "assistant": "I appreciate you coming back to share that. Boundary setting with family rarely goes smoothly the first time, especially when the dynamic has been established for years. Can you walk me through what happened? What did you say, and how did she respond? Understanding the specifics will help us refine the approach. Remember, the goal isn't a perfect outcome — it's practicing the skill."
    },
    {
        "user": "You mentioned the DEAR MAN technique for communication. Can we go through it again? I want to use it with my boss.",
        "assistant": "Of course! DEAR MAN is a DBT interpersonal effectiveness skill. It stands for: Describe the situation objectively, Express how you feel using I-statements, Assert what you need clearly, Reinforce why it benefits both parties, stay Mindful and focused on your goal, Appear confident even if you're nervous, and Negotiate if needed to find a compromise. Let's practice it with your specific situation. What do you need to communicate to your boss?"
    },
]

for pair in MEMORY_TRAINING:
    module_examples.append({"text": f"<|user|>{pair['user']}<|assistant|>{pair['assistant']}"})

print(f"Memory & context examples: {len(MEMORY_TRAINING)}")

In [ ]:
# ── Repeat module examples for emphasis (5x) ─────────────────────
# Module knowledge is critical — repeating it ensures the model
# weights shift meaningfully toward these behaviors.

REPEAT_FACTOR = 5
module_repeated = module_examples * REPEAT_FACTOR

# Mix into main training data
all_texts.extend(module_repeated)

# Re-shuffle everything
random.seed(42)
random.shuffle(all_texts)

print(f"\nModule examples (raw):     {len(module_examples)}")
print(f"Module examples (5x):      {len(module_repeated)}")
print(f"Total training samples:    {len(all_texts)}")
print(f"Module knowledge fraction: {len(module_repeated)/len(all_texts)*100:.1f}%")

In [ ]:
# ── Create HuggingFace Dataset with train/eval split ─────────────

dataset = Dataset.from_list(all_texts)
dataset = dataset.train_test_split(test_size=config.eval_split, seed=42)

print(f"Train: {len(dataset['train'])} samples")
print(f"Eval:  {len(dataset['test'])} samples")

---
## 3 — Load Phi-2 with 4-bit Quantization (QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ── 4-bit quantization config ────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4 — best for LLMs
    bnb_4bit_compute_dtype=torch.float16,  # Compute in fp16 for speed
    bnb_4bit_use_double_quant=True,        # Nested quantization — saves extra memory
)

# ── Load tokenizer ───────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    config.model,
    trust_remote_code=True,
    padding_side="right"
)
tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded: vocab_size={tokenizer.vocab_size}")
print(f"Pad token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")

In [ ]:
# ── Load model in 4-bit ──────────────────────────────────────────
print(f"Loading {config.model} in 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    config.model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

# Prepare for QLoRA training
model.config.use_cache = False  # Required for gradient checkpointing
model = prepare_model_for_kbit_training(model)

print(f"\nModel loaded. Parameters: {model.num_parameters() / 1e9:.2f}B")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# ── Apply LoRA adapters ──────────────────────────────────────────
lora_config = LoraConfig(
    r=config.lora_rank,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    target_modules=config.target_modules,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Expected output:
# trainable params: ~27M | all params: 2,780M | trainable%: ~0.97%

---
## 4 — Training with SFTTrainer

Using TRL's `SFTTrainer` for supervised fine-tuning — handles tokenization, packing, and training loop automatically.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.epochs,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.batch_size,
    gradient_accumulation_steps=config.gradient_acc_steps,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    warmup_ratio=config.warmup_ratio,
    lr_scheduler_type="cosine",
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",           # 8-bit optimizer — saves ~1GB VRAM
    gradient_checkpointing=True,
    logging_steps=config.logging_steps,
    eval_strategy="steps",
    eval_steps=config.eval_steps,
    save_strategy="steps",
    save_steps=config.save_steps,
    save_total_limit=3,                  # Keep only last 3 checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",                    # Change to "wandb" if you want tracking
    max_grad_norm=0.3,                   # Gradient clipping for stability
    group_by_length=True,                # Batch similar-length sequences
    dataloader_pin_memory=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    max_seq_length=config.context_window,
    dataset_text_field="text",
    args=training_args,
    packing=True,                        # Pack short sequences together for efficiency
)

# Verify memory before training
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU memory reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")
print(f"Ready to train.")

In [ ]:
# ── Train ────────────────────────────────────────────────────────
print("="*60)
print("  AITHER TRAINING — START")
print(f"  Model:     {config.model}")
print(f"  Datasets:  {len(DATASETS)} HF sources + Aither modules")
print(f"  Samples:   {len(all_texts)} total")
print(f"  QLoRA:     rank={config.lora_rank}, alpha={config.lora_alpha}")
print(f"  Targets:   {config.target_modules}")
print(f"  Epochs:    {config.epochs}")
print("="*60)

trainer.train()

print("\nTraining complete.")

---
## 5 — Evaluate Training Quality

In [ ]:
import math

# ── Compute final eval loss and perplexity ────────────────────────
eval_results = trainer.evaluate()
eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)

print(f"\nFinal Evaluation")
print(f"  Loss:       {eval_loss:.4f}")
print(f"  Perplexity: {perplexity:.2f}")
print(f"")
print(f"  Lower perplexity = better. A finetuned model should be")
print(f"  significantly lower than the base Phi-2 on this domain.")
print(f"  Target: perplexity < 10 on mental health/medical text.")

In [ ]:
# ── Plot training loss curve ─────────────────────────────────────
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train_steps = [x["step"] for x in logs if "loss" in x]
train_loss = [x["loss"] for x in logs if "loss" in x]
eval_steps_log = [x["step"] for x in logs if "eval_loss" in x]
eval_loss_log = [x["eval_loss"] for x in logs if "eval_loss" in x]

plt.figure(figsize=(12, 5))
plt.plot(train_steps, train_loss, label="Train Loss", alpha=0.7)
plt.plot(eval_steps_log, eval_loss_log, label="Eval Loss", marker="o", linewidth=2)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Aither Training — Loss Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nIf eval loss plateaus or increases while train loss decreases,")
print(f"the model is overfitting — consider reducing epochs or increasing data.")

---
## 6 — Test the Fine-Tuned Model

Tests cover all module knowledge areas: crisis detection, therapeutic techniques, emotional tone, and medical knowledge.

In [ ]:
# ── Generate responses to verify behavioral change ────────────────

model.config.use_cache = True
model.eval()

test_prompts = [
    # Crisis Detection (should trigger safety-aware response)
    "<|user|>I don't want to be here anymore. Everything feels hopeless.<|assistant|>",
    # Therapeutic Technique (should reference CBT/grounding)
    "<|user|>I keep having negative thoughts about myself that I can't stop. How do I deal with them?<|assistant|>",
    # Anxiety/Grounding (should reference specific techniques)
    "<|user|>I'm having a panic attack right now. Help me.<|assistant|>",
    # Medical Knowledge
    "<|user|>I've been having chest pains and shortness of breath. Should I be worried?<|assistant|>",
    # CBT Knowledge
    "<|user|>Can you explain what cognitive behavioral therapy is and how it works?<|assistant|>",
    # Depression/Behavioral Activation
    "<|user|>I can't get out of bed. I have no motivation to do anything anymore.<|assistant|>",
    # Relationship/Communication
    "<|user|>I keep getting into arguments with my partner. How can I communicate better?<|assistant|>",
    # Self-Esteem
    "<|user|>I feel worthless and like I'm not good enough for anyone.<|assistant|>",
]

print("="*60)
print("  AITHER — Post-Training Behavioral Test")
print("="*60)

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    user_msg = prompt.replace("<|user|>", "").replace("<|assistant|>", "")
    print(f"\nUser: {user_msg}")
    print(f"Aither: {response.strip()[:600]}")
    print("-"*60)

---
## 7 — Save Adapter Weights

In [ ]:
import os

# ── Save LoRA adapter weights ────────────────────────────────────
# These are SMALL (~100MB) and contain all the fine-tuning knowledge.
# To use: load base Phi-2 + these adapters.

trainer.save_model(config.output_dir)
tokenizer.save_pretrained(config.output_dir)

adapter_size = sum(
    os.path.getsize(os.path.join(config.output_dir, f))
    for f in os.listdir(config.output_dir)
    if os.path.isfile(os.path.join(config.output_dir, f))
) / 1e6

print(f"Adapter saved to: {config.output_dir}")
print(f"Adapter size: {adapter_size:.1f} MB")
print(f"\nFiles:")
for f in sorted(os.listdir(config.output_dir)):
    fpath = os.path.join(config.output_dir, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1e6
        print(f"  {f} ({size:.1f} MB)")

---
## 8 — Merge Adapter into Base Model (Full Export)

In [ ]:
# ── Merge LoRA weights back into base Phi-2 ──────────────────────
# This creates a standalone model that doesn't need PEFT at inference.
# WARNING: This requires ~6GB RAM. If you run out of memory on Colab,
# skip this cell and use the adapter weights from Step 7 instead.

from peft import AutoPeftModelForCausalLM

print("Merging LoRA adapters into base model...")
print("(This may take a few minutes)")

# Free GPU memory first
del model
del trainer
torch.cuda.empty_cache()

# Reload adapter model in fp16 on CPU for merging
merge_model = AutoPeftModelForCausalLM.from_pretrained(
    config.output_dir,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

# Merge and unload adapter layers
merged_model = merge_model.merge_and_unload()

# Save the full merged model
merged_model.save_pretrained(config.merged_dir)
tokenizer.save_pretrained(config.merged_dir)

merged_size = sum(
    os.path.getsize(os.path.join(config.merged_dir, f))
    for f in os.listdir(config.merged_dir)
    if os.path.isfile(os.path.join(config.merged_dir, f))
) / 1e9

print(f"\nMerged model saved to: {config.merged_dir}")
print(f"Total size: {merged_size:.2f} GB")
print(f"\nThis is a fully standalone Phi-2 model with Aither fine-tuning baked in.")

---
## 9 — Save Back to Aither Project

Copy trained weights to your local Aither repo structure so the project can load them directly.

In [ ]:
# ── Option A: Save to Google Drive (for transfer to local project) ─
# After training, mount Drive and copy both adapter + merged weights
# to the Aither project structure.

from google.colab import drive
import shutil

drive.mount("/content/drive")

# Save adapter weights → model/aither_trained/
adapter_drive_path = "/content/drive/MyDrive/Aither/model/aither_trained"
os.makedirs(adapter_drive_path, exist_ok=True)
shutil.copytree(config.output_dir, adapter_drive_path, dirs_exist_ok=True)
print(f"Adapter weights saved to Drive: {adapter_drive_path}")

# Save merged model → model/aither_merged/
merged_drive_path = "/content/drive/MyDrive/Aither/model/aither_merged"
os.makedirs(merged_drive_path, exist_ok=True)
shutil.copytree(config.merged_dir, merged_drive_path, dirs_exist_ok=True)
print(f"Merged model saved to Drive: {merged_drive_path}")

print(f"\nTo use locally, copy from Google Drive to your Aither project:")
print(f"  Drive/Aither/model/aither_trained/  →  model/aither_trained/")
print(f"  Drive/Aither/model/aither_merged/   →  model/aither_merged/")

In [ ]:
# ── Option B: Download adapter weights as zip (small, ~100MB) ─────
# Extract into your Aither project at: model/aither_trained/

!zip -r aither_trained.zip ./aither_trained/

from google.colab import files
files.download("aither_trained.zip")
print("\nDownloading adapter weights...")
print("Extract to: <your-project>/model/aither_trained/")

In [ ]:
# ── Option C: Push to HuggingFace Hub ────────────────────────────
# Uncomment and fill in your details to upload.

# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")
#
# merged_model.push_to_hub("your-username/aither-phi2-mental-health")
# tokenizer.push_to_hub("your-username/aither-phi2-mental-health")
# print("Pushed to HuggingFace Hub.")

---
## 10 — Loading Aither Locally (After Training)

Use this code in your local Aither project to load the fine-tuned model.
Place the adapter weights in `model/aither_trained/` or the merged model in `model/aither_merged/`.

In [ ]:
# ── Loading adapter weights locally ──────────────────────────────
# Place in your Aither project at: model/aither_trained/
#
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from peft import PeftModel
# import torch
#
# # Load base Phi-2
# base = AutoModelForCausalLM.from_pretrained(
#     "microsoft/phi-2",
#     torch_dtype=torch.float16,
#     trust_remote_code=True,
# )
#
# # Load fine-tuned Aither adapter on top
# model = PeftModel.from_pretrained(base, "./model/aither_trained")
# tokenizer = AutoTokenizer.from_pretrained("./model/aither_trained")
#
# # Generate
# prompt = "<|user|>I've been feeling anxious. Can you help?<|assistant|>"
# inputs = tokenizer(prompt, return_tensors="pt")
# outputs = model.generate(**inputs, max_new_tokens=256)
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))
#
# ── Or load the merged standalone model ──────────────────────────
# Place in your Aither project at: model/aither_merged/
#
# model = AutoModelForCausalLM.from_pretrained(
#     "./model/aither_merged",
#     torch_dtype=torch.float16,
#     trust_remote_code=True,
# )
# tokenizer = AutoTokenizer.from_pretrained("./model/aither_merged")

print("See comments above for local loading instructions.")
print("\nProject structure after training:")
print("  model/")
print("    aither_trained/    ← LoRA adapter weights (~100MB)")
print("    aither_merged/     ← Full standalone model (~5.4GB)")
print("    aither/")
print("      setup/           ← Training config & pipeline")
print("      modules/         ← Safety, RAG, Emotion, Memory")